In [ ]:
from pathlib import Path
import json
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

SEED = 42
IMG_SIZE = (224, 224)
FRAME_STEP = 5
BATCH_SIZE = 4
EPOCHS = 15
MAX_FRAMES_PER_CLASS = 300

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
categories = ["fall", "grab", "gun", "hit", "kick", 
              "lying_down", "run", "sit", "sneak", 
              "stand", "struggle", "throw", "walk"]

video_root = Path("Videos")
frame_root = Path("frames")
model_path = Path("activity_model.keras")
label_path = Path("activity_labels.json")

In [ ]:
def extract_frames(video_root: Path, frame_root: Path, categories: list[str], frame_step: int = 5, overwrite: bool = False) -> pd.DataFrame:
    records = []
    video_id = 0

    for category in categories:
        output_dir = frame_root / category
        output_dir.mkdir(parents=True, exist_ok=True)

        for video_path in sorted((video_root / category).glob("*.mp4")):
            capture = cv2.VideoCapture(str(video_path))
            frame_index = 0
            saved = 0

            while True:
                success, frame = capture.read()
                if not success:
                    break

                if frame_index % frame_step == 0:
                    output_path = output_dir / f"video_{video_id:04d}_frame_{saved:04d}.jpg"
                    if overwrite or not output_path.exists():
                        cv2.imwrite(str(output_path), frame)
                    records.append({"category": category, "video": video_path.name, "frame_path": str(output_path)})
                    saved += 1

                frame_index += 1

            capture.release()
            video_id += 1

    return pd.DataFrame(records)

frame_index_df = extract_frames(video_root, frame_root, categories, frame_step=FRAME_STEP, overwrite=False)
frame_index_df.head()

In [ ]:
frame_counts = frame_index_df["category"].value_counts().sort_index()
display(frame_counts.to_frame(name="num_frames"))

plt.figure(figsize=(12, 4))
sns.barplot(x=frame_counts.index, y=frame_counts.values, palette="crest")
plt.xticks(rotation=45, ha="right")
plt.title("Distribusi frame per kategori")
plt.ylabel("Jumlah frame")
plt.tight_layout()
plt.show()

In [ ]:
def build_frame_index(frame_root: Path, categories: list[str], max_frames_per_class: int | None = None) -> pd.DataFrame:
    records = []

    for category in categories:
        frame_paths = sorted((frame_root / category).glob("*.jpg"))
        if max_frames_per_class is not None:
            frame_paths = frame_paths[:max_frames_per_class]

        for file_path in frame_paths:
            records.append({"frame_path": str(file_path), "category": category})

    dataset_df = pd.DataFrame(records)
    if dataset_df.empty:
        raise ValueError("Dataset kosong. Pastikan folder frames sudah terisi.")

    return dataset_df

dataset_df = build_frame_index(frame_root, categories, max_frames_per_class=MAX_FRAMES_PER_CLASS)
encoder = LabelEncoder()
dataset_df["label_id"] = encoder.fit_transform(dataset_df["category"])

display(dataset_df["category"].value_counts().sort_index().to_frame(name="used_frames"))
print(f"Total samples    : {len(dataset_df)}")
print(f"Jumlah kelas     : {len(encoder.classes_)}")
print("Gambar akan dibaca secara streaming saat training, jadi tidak memenuhi RAM.")

In [ ]:
train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.30,
    random_state=SEED,
    stratify=dataset_df["label_id"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_id"],
)

num_classes = len(encoder.classes_)

print(f"Train: {len(train_df)} samples")
print(f"Val  : {len(val_df)} samples")
print(f"Test : {len(test_df)} samples")

In [ ]:
def augment_image(image: tf.Tensor) -> tf.Tensor:
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return tf.clip_by_value(image, 0.0, 1.0)

def decode_image(path: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label, depth=num_classes)
    return image, label

def make_dataset(df: pd.DataFrame, shuffle: bool = False, training: bool = False) -> tf.data.Dataset:
    paths = df["frame_path"].to_numpy(dtype=str)
    labels = df["label_id"].to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=2)
    if training:
        ds = ds.map(lambda image, label: (augment_image(image), label), num_parallel_calls=2)
    return ds.batch(BATCH_SIZE).prefetch(1)

train_ds = make_dataset(train_df, shuffle=True, training=True)

val_ds = make_dataset(val_df)

test_ds = make_dataset(test_df)

In [ ]:
try:
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet",
    )
    print("Menggunakan pretrained ImageNet weights.")
except Exception as exc:
    print(f"ImageNet weights tidak bisa dimuat: {exc}")
    print("Fallback ke MobileNetV2 tanpa pretrained weights.")
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights=None,
    )

base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(model_path, monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

history_df = pd.DataFrame(history.history)
history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")

axes[0].set_xlabel("Epoch")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

In [ ]:
best_model = tf.keras.models.load_model(model_path)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")

In [ ]:
y_prob = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
target_names = encoder.classes_.tolist()

y_true = test_df["label_id"].to_numpy()

print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
label_path.write_text(json.dumps(target_names, indent=2), encoding="utf-8")
print(f"Model saved to: {model_path}")
print(f"Labels saved to: {label_path}")